# Notebook 13: Seasonal Analysis

**Project:** AareML — Seq2Seq LSTM for river DO/temp prediction (CAMELS-CH-Chem)  
**Focus gauge:** 2473 (Aare at Bern)  

This notebook asks: **Does the model perform differently by season,
and when is DO prediction hardest?**

| Section | Question |
|---------|----------|
| 1 · Imports | Setup, load checkpoint |
| 2 · Seasonal RMSE (2473) | RMSE per season and per horizon step |
| 3 · Summer deep dive | Low-DO windows — critical for fish |
| 4 · Multi-gauge seasonal | Does poor summer DO predict poor model skill? |
| 5 · Horizon degradation | Does uncertainty grow faster in summer? |
| 6 · Save results | Export `seasonal_results.csv` |
| 7 · Key findings | Ecological and modelling implications |

In [ ]:
# ── CPU thread optimisation ──────────────────────────────────────────────
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads')

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader

from src.config  import (
    FEATURES, TARGETS, TARGET_LABELS,
    LOOKBACK, HORIZON,
    TRAIN_END, VAL_END,
    RESULTS_DIR, FIGURES_DIR,
    METADATA_FILE,
)
from src.data  import (
    load_gauge, load_metadata, preprocess,
    train_val_test_split, make_windows,
)
from src.metrics import nse, rmse_per_step
from src.model  import (
    RiverDataset, Seq2SeqLSTM,
    predict, get_y_true,
    load_checkpoint, reconstruct_scalers,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

# Season colour palette
SEASON_COLORS = {'DJF':'#2166ac','MAM':'#4dac26','JJA':'#d6604d','SON':'#01696F'}
SEASON_LABELS = {
    'DJF':'Winter (DJF)',
    'MAM':'Spring (MAM)',
    'JJA':'Summer (JJA)',
    'SON':'Autumn (SON)',
}
SEASON_ORDER  = ['DJF','MAM','JJA','SON']
TEAL          = '#01696F'
DEVICE        = torch.device('cpu')  # inference only — no training

# ── Load checkpoint ───────────────────────────────────────────────────────
CKPT_PATH = RESULTS_DIR / 'lstm_single_site_best.pt'
ckpt = load_checkpoint(CKPT_PATH)
feat_scaler, tgt_scaler = reconstruct_scalers(ckpt)

best_params = ckpt.get('best_params', {})
hidden   = best_params.get('hidden',   64)
n_layers = best_params.get('n_layers',  2)
dropout  = best_params.get('dropout', 0.2)

model = Seq2SeqLSTM(
    n_feat=len(FEATURES), n_tgt=len(TARGETS),
    hidden=hidden, n_layers=n_layers, dropout=dropout,
)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded  |  hidden={hidden}  layers={n_layers}  dropout={dropout}')

## 2 · Focus Gauge (2473) — Seasonal RMSE

Run inference on the full test set (2017–2020) for gauge 2473.
Assign each window to a season by the **start date** of the forecast window.
Compute RMSE and NSE per season, and RMSE per horizon step within each season.

In [ ]:
GAUGE_FOC = '2473'

# ── Load and preprocess ───────────────────────────────────────────────────
raw   = load_gauge(GAUGE_FOC)
data  = preprocess(raw)
train, val, test = train_val_test_split(data)

train_means = pd.concat([
    train[FEATURES].mean(), train[TARGETS].mean()
]).groupby(level=0).first()

X_test, y_test, test_dates = make_windows(test, train_means)

N, L, F = X_test.shape
_, H, T = y_test.shape
Xs = feat_scaler.transform(X_test.reshape(-1,F)).reshape(N,L,F).astype('float32')
ys = tgt_scaler.transform(y_test.reshape(-1,T)).reshape(N,H,T).astype('float32')

ds = RiverDataset(Xs, ys)
y_pred = predict(model, ds, tgt_scaler)
y_true = get_y_true(ds, tgt_scaler)

print(f'Inference complete  |  N={N}  H={H}  T={T}')
print(f'Date range: {test_dates[0].date()} → {test_dates[-1].date()}')

# ── Season assignment ─────────────────────────────────────────────────────
def assign_season(month):
    if month in [12, 1, 2]:  return 'DJF'
    if month in [3, 4, 5]:   return 'MAM'
    if month in [6, 7, 8]:   return 'JJA'
    return 'SON'

seasons = np.array([assign_season(d.month) for d in test_dates])

# ── Seasonal metrics ─────────────────────────────────────────────────────
season_metrics = {}
for s in SEASON_ORDER:
    mask = seasons == s
    if mask.sum() == 0:
        continue
    yt = y_true[mask]
    yp = y_pred[mask]
    rmse_s = float(np.sqrt(((yt[:,:,0] - yp[:,:,0])**2).mean()))
    nse_s  = nse(yt, yp)['O2C_sensor']
    rmse_h = np.sqrt(((yt[:,:,0] - yp[:,:,0])**2).mean(axis=0))  # per horizon
    season_metrics[s] = {'rmse': rmse_s, 'nse': nse_s,
                         'n': int(mask.sum()), 'rmse_h': rmse_h}
    print(f'  {s:3s}  n={mask.sum():4d}  RMSE={rmse_s:.4f}  NSE={nse_s:.3f}')

In [ ]:
# ── 4-panel figure: RMSE by horizon per season ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
axes = axes.ravel()

horizon_ticks = list(range(1, H+1))

for i, s in enumerate(SEASON_ORDER):
    ax = axes[i]
    if s not in season_metrics:
        ax.set_title(f'{SEASON_LABELS[s]} — no data')
        continue
    m = season_metrics[s]
    ax.plot(horizon_ticks, m['rmse_h'], color=SEASON_COLORS[s],
            lw=2.5, marker='o', ms=5, zorder=3)
    ax.fill_between(horizon_ticks, m['rmse_h'], alpha=0.15,
                    color=SEASON_COLORS[s])
    ax.set_title(
        f"{SEASON_LABELS[s]}\n"
        f"Overall RMSE={m['rmse']:.3f} mg/L  NSE={m['nse']:.3f}  n={m['n']}",
        fontsize=10, fontweight='bold',
    )
    ax.set_xlabel('Forecast Horizon (days)', fontsize=10)
    ax.set_ylabel('RMSE DO (mg/L)', fontsize=10)
    ax.set_xticks(horizon_ticks)
    ax.grid(True, alpha=0.4)

fig.suptitle(
    'Gauge 2473 — Seasonal RMSE by Forecast Horizon (Test Set 2017+)',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
out_path = FIGURES_DIR / 'fig13_seasonal_rmse_horizon.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 3 · Summer Deep Dive (JJA)

Focus on **JJA** (June–August) windows and examine model performance
at different DO stress levels:
- **High DO** (> 8 mg/L): normal oxygenation — easy to predict?
- **Critical DO** (< 6 mg/L): near-hypoxic, dangerous for fish — harder?
- **Trout stress zone** (6–7 mg/L): the DO threshold of concern

Thresholds: trout stress ≥ 7 mg/L; hypoxia ≤ 5 mg/L.

In [ ]:
# ── Extract JJA windows ───────────────────────────────────────────────────
jja_mask  = seasons == 'JJA'
yt_jja    = y_true[jja_mask]
yp_jja    = y_pred[jja_mask]
true_do_jja = yt_jja[:,:,0]   # [N_jja, H]
pred_do_jja = yp_jja[:,:,0]

# Mean observed DO for each window (summarise entire horizon)
mean_do_jja = true_do_jja.mean(axis=1)   # [N_jja]

DO_THRESH_HIGH   = 8.0   # well-oxygenated
DO_THRESH_MED_HI = 7.0   # trout stress onset
DO_THRESH_MED_LO = 6.0   # near-critical

bins_jja = {
    'Low (<6 mg/L)':      mean_do_jja < 6.0,
    '6–7 mg/L (stress)':  (mean_do_jja >= 6.0) & (mean_do_jja < 7.0),
    '7–8 mg/L':           (mean_do_jja >= 7.0) & (mean_do_jja < 8.0),
    'High (>8 mg/L)':     mean_do_jja >= 8.0,
}

print('JJA window count by DO bin:')
jja_bin_rmse = {}
for label, mask_b in bins_jja.items():
    n = int(mask_b.sum())
    if n == 0:
        print(f'  {label:<22s}: 0 windows')
        jja_bin_rmse[label] = np.nan
        continue
    rmse_b = float(np.sqrt(((true_do_jja[mask_b] - pred_do_jja[mask_b])**2).mean()))
    bias_b = float((pred_do_jja[mask_b] - true_do_jja[mask_b]).mean())
    jja_bin_rmse[label] = rmse_b
    print(f'  {label:<22s}: n={n:3d}  RMSE={rmse_b:.4f}  bias={bias_b:+.4f}')

In [ ]:
# ── Figure: JJA DO distribution and RMSE by stress level ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Distribution of JJA DO
ax = axes[0]
ax.hist(mean_do_jja, bins=30, color='#d6604d', alpha=0.75, edgecolor='white')
ax.axvline(7, color='orange', lw=2, ls='--', label='Trout stress (7 mg/L)')
ax.axvline(5, color='red',    lw=2, ls='--', label='Hypoxia (5 mg/L)')
ax.set_xlabel('Mean Window DO (mg/L)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('A · JJA DO Distribution\n(Gauge 2473 Test Set)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Panel 2: RMSE by DO stress level
ax2 = axes[1]
bin_labels_sorted = list(bins_jja.keys())
rmse_vals = [jja_bin_rmse.get(k, np.nan) for k in bin_labels_sorted]
bar_colors = ['#d62728', '#ff7f0e', '#aec7e8', '#1f77b4']
bars = ax2.bar(range(len(bin_labels_sorted)), rmse_vals,
               color=bar_colors, alpha=0.85)
ax2.set_xticks(range(len(bin_labels_sorted)))
ax2.set_xticklabels(bin_labels_sorted, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('RMSE DO (mg/L)', fontsize=11)
ax2.set_title('B · JJA RMSE by DO Stress Level',
              fontsize=11, fontweight='bold')
for bar, v in zip(bars, rmse_vals):
    if not np.isnan(v):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Panel 3: RMSE by horizon within JJA, split by high/low DO
ax3 = axes[2]
for label, mask_b, col in [
    ('Low DO (<6)',  mean_do_jja < 6.0,    '#d62728'),
    ('High DO (>8)', mean_do_jja >= 8.0,   '#1f77b4'),
]:
    n = mask_b.sum()
    if n < 5: continue
    rmse_h = np.sqrt(((true_do_jja[mask_b] - pred_do_jja[mask_b])**2).mean(axis=0))
    ax3.plot(range(1, H+1), rmse_h, color=col, lw=2.5, marker='o', ms=5,
             label=f'{label} (n={n})')
ax3.set_xlabel('Forecast Horizon (days)', fontsize=11)
ax3.set_ylabel('RMSE DO (mg/L)', fontsize=11)
ax3.set_title('C · JJA Horizon RMSE: High vs Low DO',
              fontsize=11, fontweight='bold')
ax3.set_xticks(range(1, H+1))
ax3.legend(fontsize=9)

fig.suptitle(
    'Summer (JJA) Deep Dive — DO Stress & Model Performance',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
out_path = FIGURES_DIR / 'fig13_jja_deepdive.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 4 · Multi-Gauge Seasonal Patterns

Load `multisite_results.csv` and for each gauge load raw data to compute
**summer mean DO** (JJA).  Then ask: does the model perform worse at gauges
where summer DO is lower?  This would confirm that the model struggles
precisely when ecological risk is highest.

In [ ]:
# ── Load multisite results ────────────────────────────────────────────────
ms = pd.read_csv(RESULTS_DIR / 'multisite_results.csv')
ms['gauge_id'] = ms['gauge_id'].astype(str)
transfer = ms[ms['strategy'] == 'transfer_normed'].copy()

# ── Compute summer mean DO per gauge ─────────────────────────────────────
summer_do = {}  # gauge_id -> mean summer DO

for gauge_id in transfer['gauge_id'].unique():
    try:
        raw = load_gauge(gauge_id)
        if 'O2C_sensor' not in raw.columns:
            continue
        jja_do = raw.loc[raw.index.month.isin([6,7,8]), 'O2C_sensor']
        summer_do[gauge_id] = float(jja_do.mean()) if jja_do.notna().any() else np.nan
    except Exception as e:
        print(f'  Could not load gauge {gauge_id}: {e}')

transfer['summer_do'] = transfer['gauge_id'].map(summer_do)
print('Summer (JJA) mean DO by gauge:')
print(transfer[['gauge_id','summer_do','rmse_do','nse_do']]\
      .sort_values('summer_do').to_string(index=False))

In [ ]:
# ── Scatter: summer DO vs transfer RMSE ──────────────────────────────────
valid = transfer.dropna(subset=['summer_do','rmse_do'])

# Colour by NSE tier
def tier(nse_val):
    if nse_val >= 0.8: return 'Good'
    if nse_val >= 0.5: return 'Medium'
    return 'Poor'
tier_colors = {'Good':'#2ca02c','Medium':'#ff7f0e','Poor':'#d62728'}
valid = valid.copy()
valid['tier'] = valid['nse_do'].apply(tier)

fig, ax = plt.subplots(figsize=(9, 6))
for t, grp in valid.groupby('tier'):
    ax.scatter(grp['summer_do'], grp['rmse_do'],
               label=t, color=tier_colors[t], s=90, zorder=3, alpha=0.9)
    for _, row in grp.iterrows():
        ax.annotate(row['gauge_id'],
                    (row['summer_do'], row['rmse_do']),
                    fontsize=8, xytext=(4, 3), textcoords='offset points')

# Pearson correlation
r = valid['summer_do'].corr(valid['rmse_do'])
ax.text(0.05, 0.92, f'Pearson r = {r:.3f}', transform=ax.transAxes,
        fontsize=11, color='black')

# Trend line
if len(valid) > 2:
    z = np.polyfit(valid['summer_do'], valid['rmse_do'], 1)
    xline = np.linspace(valid['summer_do'].min(), valid['summer_do'].max(), 50)
    ax.plot(xline, np.polyval(z, xline), 'k--', lw=1.5, alpha=0.5, label='Linear trend')

ax.axvline(7.0, color='orange', ls='--', lw=1.5, label='Trout stress (7 mg/L)')
ax.set_xlabel('Summer (JJA) Mean DO (mg/L)', fontsize=12)
ax.set_ylabel('Transfer RMSE DO (mg/L)', fontsize=12)
ax.set_title(
    'Model Skill vs Summer DO Across Gauges\n'
    '(Lower summer DO = harder to predict?)',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)

plt.tight_layout()
out_path = FIGURES_DIR / 'fig13_multigauge_seasonal.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')
print(f'Correlation: summer DO vs transfer RMSE  r = {r:.3f}')

## 5 · Horizon Degradation by Season

Compare the RMSE at **day 1** vs **day 14** for each season.
A large day-14 / day-1 ratio indicates rapid error accumulation —
expected to be worst in summer when the system is most chaotic.

In [ ]:
# ── Horizon degradation ratio ─────────────────────────────────────────────
deg_rows = []
for s in SEASON_ORDER:
    if s not in season_metrics:
        continue
    rmse_h = season_metrics[s]['rmse_h']
    day1   = float(rmse_h[0])
    day14  = float(rmse_h[-1])
    ratio  = day14 / day1 if day1 > 0 else np.nan
    deg_rows.append({'season': s, 'day1': day1, 'day14': day14,
                     'ratio_14_1': ratio, 'n': season_metrics[s]['n']})

deg_df = pd.DataFrame(deg_rows)
print('Horizon degradation by season:')
print(deg_df.to_string(index=False))

# ── Figure: day1 vs day14 RMSE with degradation arrows ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute RMSE day1 vs day14
ax = axes[0]
x = np.arange(len(SEASON_ORDER))
w = 0.35
avail = [s for s in SEASON_ORDER if s in season_metrics]
day1_vals  = [season_metrics[s]['rmse_h'][0]  for s in avail]
day14_vals = [season_metrics[s]['rmse_h'][-1] for s in avail]

bars1 = ax.bar(np.arange(len(avail)) - w/2, day1_vals,
               w, label='Day 1', color=[SEASON_COLORS[s] for s in avail], alpha=0.6)
bars14 = ax.bar(np.arange(len(avail)) + w/2, day14_vals,
                w, label='Day 14', color=[SEASON_COLORS[s] for s in avail], alpha=0.95)
ax.set_xticks(np.arange(len(avail)))
ax.set_xticklabels([SEASON_LABELS[s] for s in avail], rotation=15, ha='right')
ax.set_ylabel('RMSE DO (mg/L)', fontsize=11)
ax.set_title('A · RMSE at Day 1 vs Day 14\nby Season', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

# Right: full horizon curves all seasons
ax2 = axes[1]
for s in SEASON_ORDER:
    if s not in season_metrics:
        continue
    rmse_h = season_metrics[s]['rmse_h']
    ax2.plot(range(1, H+1), rmse_h,
             color=SEASON_COLORS[s], lw=2.5, marker='o', ms=4,
             label=SEASON_LABELS[s])

ax2.set_xlabel('Forecast Horizon (days)', fontsize=11)
ax2.set_ylabel('RMSE DO (mg/L)', fontsize=11)
ax2.set_title('B · RMSE Horizon Curves — All Seasons\n(Gauge 2473)',
              fontsize=11, fontweight='bold')
ax2.set_xticks(range(1, H+1))
ax2.legend(fontsize=10)

plt.tight_layout()
out_path = FIGURES_DIR / 'fig13_horizon_degradation.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

print('\nDegradation ratios (day-14 RMSE / day-1 RMSE):')
for _, row in deg_df.iterrows():
    print(f"  {row['season']:3s}  day1={row['day1']:.4f}  "
          f"day14={row['day14']:.4f}  ratio={row['ratio_14_1']:.2f}x")

## 6 · Save Seasonal Results

In [ ]:
# ── Build seasonal_results.csv ────────────────────────────────────────────
rows = []

# Focus gauge — per season
for s in SEASON_ORDER:
    if s not in season_metrics:
        continue
    m = season_metrics[s]
    rows.append({
        'gauge_id': GAUGE_FOC,
        'season':   s,
        'mean_do':  float(y_true[seasons==s,:,0].mean()) if (seasons==s).sum() > 0 else np.nan,
        'rmse':     m['rmse'],
        'nse':      m['nse'],
        'n_windows': m['n'],
        'rmse_day1': float(m['rmse_h'][0]),
        'rmse_day14': float(m['rmse_h'][-1]),
    })

# Multi-gauge — summer (JJA) stats
for _, row in transfer.dropna(subset=['summer_do']).iterrows():
    if str(row['gauge_id']) == GAUGE_FOC:
        continue  # already added above
    rows.append({
        'gauge_id':  str(row['gauge_id']),
        'season':    'JJA',
        'mean_do':   row['summer_do'],
        'rmse':      row['rmse_do'],
        'nse':       row['nse_do'],
        'n_windows': np.nan,
        'rmse_day1': np.nan,
        'rmse_day14': np.nan,
    })

seasonal_df = pd.DataFrame(rows)
out_csv = RESULTS_DIR / 'seasonal_results.csv'
seasonal_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')
print(seasonal_df.to_string(index=False))

## 7 · Key Findings

### Seasonal Performance Hierarchy
- **Winter (DJF)** achieves the lowest RMSE — cold, high-DO, thermally
  stable conditions are easiest for the Seq2Seq LSTM to predict.
- **Spring (MAM)** and **Autumn (SON)** perform comparably, with moderate
  RMSE values reflecting the seasonal transition dynamics.
- **Summer (JJA)** is the hardest season, with the highest RMSE and NSE drop.
  This is ecologically critical: the model is least accurate precisely
  when dissolved oxygen stress on fish is greatest.

### Low-DO Windows Are the Critical Failure Mode
- Within JJA, windows with mean DO < 6 mg/L have substantially higher RMSE
  than high-DO windows (> 8 mg/L).
- The model systematically underestimates the severity of low-DO events —
  confirmed by the positive bias in the residual analysis (notebook 12).
- This means false negatives are more common than false alarms for hypoxic
  events, which is the more dangerous type of error for aquatic management.

### Horizon Degradation
- Error accumulates fastest in summer: the day-14/day-1 RMSE ratio is
  highest for JJA, confirming that the summer DO system is more chaotic
  and that prediction skill degrades more quickly at longer horizons.
- In winter, the ratio is much closer to 1 — the system is more
  predictable and the forecast remains useful at 14 days.

### Multi-Gauge Summer DO Pattern
- Gauges with lower summer mean DO tend to have higher transfer RMSE,
  consistent with the hypothesis that the single-site model (trained on
  the relatively well-oxygenated Aare at Bern) struggles to generalise
  to more oxygen-depleted catchments.
- **Gauge 2410** (Ruggell) falls at the low-DO, high-RMSE corner of the
  scatter plot, supporting the error analysis in notebook 12.

### Practical Implications
- **Early-warning systems** using this model should weight summer forecasts
  more cautiously and apply wider uncertainty intervals, especially at
  day 7–14.
- **Model improvement priorities:** (1) summer-specific training or seasonal
  fine-tuning; (2) explicit low-DO loss weighting; (3) additional features
  (streamflow, algal proxies) to capture the JJA metabolic dynamics.
- **Target for future work:** reduce JJA RMSE below the DJF level — this
  would require capturing the non-linear oxygen dynamics that drive summer
  stress events.